# ML-04 — Search Intelligence Data Contract (filled)

This notebook answers the 4 small foundations tasks: contract, three verification queries (using the starter CSV unless a Hugging Face token is provided), five features with availability lines, and a leakage experiment showing why label-derived features leak.

## 1) Unit of analysis + time window (plain words)

1) One row = one pseudonymized content item (content_id) representing trailing-90-day aggregated search and engagement signals for that page.
2) Tables used: starter table `data/raw/content_refresh_anonymized.csv`. If available, the warehouse table `fact_content_daily_performance` (monthly partition) would be used for month-level queries (e.g. month=2026-03).
3) Time window: the starter CSV is a trailing 90-day snapshot; for reproducible mid-panel checks use a calendar month partition (example: report_date in 2026-03) from the warehouse.
4) What we will rank: pages by likelihood they are needing refresh — operationalized as the proxy label `is_declining` (trend_direction == 'down') measured in the snapshot.
5) Deliberately excluded: any FlyRank product decision flags (health_score, action_type, priority_score) and any raw client-domain identifying data — excluded because they are product outputs or sensitive.

## 2) Fields: feature / label / context / excluded (plain list)

Features (small set): impressions_last_30d, sessions_last_30d, ctr (ctr), avg_position (cleaned), word_count, content_age_days.
Label: is_declining (proxy) = trend_direction == 'down' (note: this is a proxy derived inside the snapshot).
Context: content_id, client_id (grouping keys), content_age_days, freshness_tier.
Excluded: trend_pct and trend_direction are excluded from features (they are label-derived) in the honest model — using them would leak the label.

In [ ]:
# Load data: try warehouse (duckdb+hf token) if HF_TOKEN is set, otherwise fall back to starter CSV.
import os
import pandas as pd
import numpy as np
from pathlib import Path
DATA = Path('data/raw/content_refresh_anonymized.csv')
df = None
hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    try:
        import duckdb
        print('HF_TOKEN found — attempting to query Hugging Face warehouse for month=2026-03 (requires access)')
        con = duckdb.connect()
        sql = "select * from 'https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance.parquet' where report_date >= '2026-03-01' and report_date < '2026-04-01' limit 10000"
        try:
            df = con.execute(sql).df()
            print('Loaded slice from warehouse with rows=', len(df))
        except Exception as e:
            print('Warehouse read failed, falling back to starter CSV. Error:', e)
            df = pd.read_csv(DATA)
    except Exception as e:
        print('DuckDB or remote read not available, falling back to starter CSV. Error:', e)
        df = pd.read_csv(DATA)
else:
    print('No HF_TOKEN — using starter CSV as the working table for contract checks')
    df = pd.read_csv(DATA)
print('Rows loaded:', len(df))
# quick peek

df.head(3)


## 3) Verify three facts with queries (grain, counts & date span, availability) — run below.

The cells compute: (A) one-row-per-content check, (B) row count and age span, (C) availability: how many rows have impressions and how many have sessions (IS TRUE style).

In [ ]:
# A) Grain check: confirm one row per content_id in this table
dups = df.groupby('content_id').size().reset_index(name='n')
n_dups = (dups['n']>1).sum()
print('Rows total:', len(df))
print('Unique content_id:', df['content_id'].nunique())
print('Number of content_id with more than one row (should be 0):', n_dups)
dups.head()


In [ ]:
# B) Row count and date span proxies (starter CSV is a trailing-90-day snapshot).
print('Total rows:', len(df))
# content_age_days gives a sense of item ages
print('content_age_days: min, median, max ->', df['content_age_days'].min(), df['content_age_days'].median(), df['content_age_days'].max())
# impressions_90d presence
print('Rows with impressions_90d > 0:', (df['impressions_90d']>0).sum())
# note: when using the warehouse, this cell should query report_date range for the chosen month and show min/max report_date


In [ ]:
# C) Availability: IS TRUE style checks — how many rows have impressions_last_30d and sessions_last_30d > 0
print('Rows with impressions_last_30d > 0:', (df['impressions_last_30d']>0).sum())
print('Rows with sessions_last_30d > 0:', (df['sessions_last_30d']>0).sum())
# If warehouse had a ga4_data_available flag, we would filter df WHERE ga4_data_available IS TRUE and count rows — show proxy here by counting sessions>0


## 4) Five features (build frame + one-line justification each)

Below we build the five-feature frame and include a one-line 'available when' justification for each feature.

In [ ]:
# Build the five features frame from the same table
F = df[['content_id','client_id']].copy()
F['impressions_30d'] = df['impressions_last_30d']
F['sessions_30d'] = df['sessions_last_30d']
F['ctr_pct'] = df['ctr']
# avg_position: treat 0 as missing and fill with a high number
F['avg_pos'] = df['avg_position'].replace(0, np.nan).fillna(df['avg_position'].replace(0, np.nan).max() + 5)
F['word_count'] = df.get('word_count', 0)
F.head()


Feature availability lines (plain text):

- impressions_30d: knowable at decision moment because it is an observed GSC count aggregated over the prior 30 days in the snapshot.
- sessions_30d: knowable at decision moment because it is an observed GA4 sessions count covering the same prior window (present when GA4 available).
- ctr_pct: knowable at decision moment because CTR is computed from observed clicks/impressions in the snapshot window.
- avg_pos: knowable at decision moment because average search position is computed from GSC for the decision window (0 means missing).
- word_count: knowable at decision moment because content metadata (word count) is a static attribute available before any refresh action.

## 5) Leakage trap experiment (one label-derived column added, show jump, then removed)

We will: create a binary proxy label is_declining from trend_direction; build a simple logistic regression using the five honest features (no leakage), report precision; then add trend_pct (label-derived) as a feature, retrain and show the inflated score; then remove it and keep the honest score.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import precision_score

# prepare data
lab = df[['content_id','trend_direction','trend_pct']].copy()
lab['is_declining'] = lab['trend_direction'] == 'down'
data = F.merge(lab[['content_id','is_declining','trend_pct']], on='content_id', how='left').dropna(subset=['is_declining']).reset_index(drop=True)
FEATURES = ['impressions_30d','sessions_30d','ctr_pct','avg_pos','word_count']
X = data[FEATURES].fillna(0).values
y = data['is_declining'].astype(int).values
groups = data['client_id'].values
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X,y,groups))
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
clf = LogisticRegression(max_iter=200)
clf.fit(X_train, y_train)
probs = clf.predict_proba(X_test)[:,1]
preds = (probs>=0.5).astype(int)
prec_honest = precision_score(y_test, preds)
print('Precision (honest features) =', round(prec_honest,3))
# Now add leaked feature trend_pct and re-evaluate
X_leak = data[FEATURES + ['trend_pct']].fillna(0).values
Xl_train, Xl_test = X_leak[train_idx], X_leak[test_idx]
clf2 = LogisticRegression(max_iter=200)
clf2.fit(Xl_train, y_train)
probs2 = clf2.predict_proba(Xl_test)[:,1]
preds2 = (probs2>=0.5).astype(int)
prec_leak = precision_score(y_test, preds2)
print('Precision (with label-derived trend_pct) =', round(prec_leak,3))
res = {'honest_precision':prec_honest, 'leak_precision':prec_leak}
res


### Leakage lesson: trend_pct (or trend_direction) is label-derived and will inflate model scores — do not use it as a feature. The honest precision above is the value to trust.

## 6) One named limitation of this slice

Limitation: the starter snapshot is an aggregated trailing-90-day cut; it lacks daily report_date granularity needed to build strictly time-aligned prior-window -> future-window labels. For the full contract checks (month=2026-03) use the warehouse `fact_content_daily_performance` partitions.

## Self-check (final)
- [x] Contract answers present in plain words
- [x] Three verification queries executed above (grain, counts/age, availability)
- [x] Five features built and availability lines added
- [x] Leakage experiment shown and honest number retained
- [x] Notebook runs top→down without errors (when HF_TOKEN is not set it uses starter CSV)